# Stacking — a meta-learner over base models

> Tutorial pair for [`stacking.py`](stacking.py).

## 1. Intuition
Different models make different mistakes. Instead of *averaging* them (voting) or
resampling *one* model (bagging), **stacking trains a second model to learn how
to combine the first ones**. The base models output predictions; a *meta-learner*
takes those predictions as its input features and learns the best blend — e.g.
"trust the tree near the boundary, the logistic model elsewhere." The one catch:
the meta-features must be generated **out-of-fold** so the meta-learner never
sees a base model predicting its own training data.

## 2. Concept (the slide)
- **Level-0 (base) models:** heterogeneous learners (tree, logistic regression,
  k-NN, …) trained on the data.
- **Level-1 (meta) model:** trained on the base models' predictions as features.
- **Out-of-fold (OOF) trick:** generate each row's meta-features from base models
  fit on the *other* folds → no leakage. (If you used in-sample base predictions,
  the meta-learner would over-trust overfit base models.)
- **Passthrough:** optionally also feed the original features to the meta-learner.
- At test time, base models are refit on the full training set.

## 3. Math derivation

**The leakage problem, formally.** Suppose base model $g$ overfits: on its own
training data $\hat g(x_i)\approx y_i$ even though it generalizes poorly. If the
meta-learner $f$ is trained on features $z_i=\hat g(x_i)$ computed *in-sample*,
those $z_i$ look almost perfect, so $f$ learns to put all its weight on $g$ —
and then fails at test time, where $g$ is no longer near-perfect. The meta-model
must see base predictions whose error distribution **matches test time**.

**Out-of-fold construction (Wolpert).** Partition the $n$ rows into $K$ folds
$\{V_1,\dots,V_K\}$. For each base model $g^{(b)}$ and each fold $k$:

$$g^{(b)}_{-k}=\text{fit on } \{(x_i,y_i): i\notin V_k\},\qquad
  z_i^{(b)}=g^{(b)}_{-k}(x_i)\ \text{ for } i\in V_k .$$

Every meta-feature $z_i^{(b)}$ thus comes from a model that **did not train on
row $i$**, so it carries an honest, test-like error. Stack the columns into
$Z\in\mathbb R^{n\times(B\cdot c)}$ ($c=1$ for regression, $K{-}1$ probability
columns per base for $K$-class classification — dropping one column avoids the
sum-to-one collinearity). Train the meta-learner

$$f^\star=\arg\min_f \sum_i \ell\big(y_i,\, f(z_i)\big).$$

**Test time.** Refit each base model on **all** $n$ rows (more data ⇒ better base
predictions), form $z(x)=\big(g^{(1)}(x),\dots,g^{(B)}(x)\big)$, and predict
$f^\star(z(x))$.

**Why it can beat any single base model.** Stacking can implement a *convex
combination* of base predictions (if $f$ is e.g. linear/logistic with positive
weights), so in the worst case it recovers the best single model; with diverse,
decorrelated bases it does strictly better — it is doing supervised model
selection *per region of feature space*. Using a **simple** meta-learner (linear,
ridge, logistic) is standard: the bases already did the heavy lifting, and a
simple blender resists overfitting the small meta-feature set.

## 4. NumPy implementation (OOF meta-features + meta-learner)

In [ ]:
# ===== actual implementation from stacking.py =====
from __future__ import annotations

import copy

import numpy as np

SEED = 0

class _LogReg:
    """Multinomial logistic regression (softmax) via full-batch gradient descent."""

    def __init__(self, lr=0.1, n_iter=400, l2=1e-3):
        self.lr, self.n_iter, self.l2 = lr, n_iter, l2

    @staticmethod
    def _softmax(Z):
        Z = Z - Z.max(1, keepdims=True)
        E = np.exp(Z)
        return E / E.sum(1, keepdims=True)

    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y).astype(int)
        n, d = X.shape
        self.classes_ = np.unique(y)
        K = len(self.classes_)
        Y = np.eye(K)[np.searchsorted(self.classes_, y)]   # one-hot
        self.W = np.zeros((d, K)); self.b = np.zeros(K)
        for _ in range(self.n_iter):
            P = self._softmax(X @ self.W + self.b)
            gW = X.T @ (P - Y) / n + self.l2 * self.W
            gb = (P - Y).mean(0)
            self.W -= self.lr * gW; self.b -= self.lr * gb
        return self

    def predict_proba(self, X):
        return self._softmax(np.asarray(X, float) @ self.W + self.b)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]

class _Tree:
    """Shallow CART (depth-limited) base learner — diverse from logistic reg.

    Leaves store class proportions, so it exposes `predict_proba` for stacking.
    Vectorized prefix-sum split search keeps it fast."""

    def __init__(self, task="classification", max_depth=4, min_samples_split=2):
        self.task, self.max_depth, self.min_samples_split = task, max_depth, min_samples_split

    def _best_split(self, X, y):
        n, d = X.shape
        best = (np.inf, None, None)
        for f in range(d):
            order = np.argsort(X[:, f], kind="mergesort")
            xs = X[order, f]
            valid = xs[:-1] != xs[1:]
            if not valid.any():
                continue
            cl_n = np.arange(1, n); cr_n = n - cl_n
            if self.task == "classification":
                oh = np.eye(self._K)[y[order].astype(int)]
                cum = np.cumsum(oh, axis=0); tot = cum[-1]
                cl, cr = cum[:-1], tot - cum[:-1]
                gl = 1 - ((cl / cl_n[:, None]) ** 2).sum(1)
                gr = 1 - ((cr / cr_n[:, None]) ** 2).sum(1)
                s = (cl_n * gl + cr_n * gr) / n
            else:
                ys = y[order].astype(float)
                cs = np.cumsum(ys)[:-1]; cs2 = np.cumsum(ys ** 2)[:-1]
                tot, tot2 = cs[-1] + ys[-1], cs2[-1] + ys[-1] ** 2
                vl = cs2 / cl_n - (cs / cl_n) ** 2
                vr = (tot2 - cs2) / cr_n - ((tot - cs) / cr_n) ** 2
                s = (cl_n * vl + cr_n * vr) / n
            s = np.where(valid, s, np.inf)
            j = int(np.argmin(s))
            if s[j] < best[0]:
                best = (s[j], f, (xs[j] + xs[j + 1]) / 2)
        return best

    def _leaf(self, y):
        if self.task == "classification":
            return np.bincount(y.astype(int), minlength=self._K) / len(y)
        return float(y.mean())

    def _build(self, X, y, depth):
        if (len(y) < self.min_samples_split or depth >= self.max_depth or
                len(np.unique(y)) == 1):
            return ("leaf", self._leaf(y))
        _, f, t = self._best_split(X, y)
        if f is None:
            return ("leaf", self._leaf(y))
        m = X[:, f] <= t
        return ("node", f, t, self._build(X[m], y[m], depth + 1),
                self._build(X[~m], y[~m], depth + 1))

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y)
        if self.task == "classification":
            self.classes_ = np.unique(y)
            self._K = int(y.max()) + 1
        self.root = self._build(X, y, 0)
        return self

    def _one(self, x, node):
        if node[0] == "leaf":
            return node[1]
        _, f, t, l, r = node
        return self._one(x, l if x[f] <= t else r)

    def predict_proba(self, X):
        return np.array([self._one(x, self.root) for x in np.asarray(X, float)])

    def predict(self, X):
        if self.task == "classification":
            return self.classes_[self.predict_proba(X).argmax(1)]
        return np.array([self._one(x, self.root) for x in np.asarray(X, float)])

def _kfold_indices(n, k, rng):
    idx = rng.permutation(n)
    return np.array_split(idx, k)

class _Ridge:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y, float)
        Xb = np.hstack([np.ones((len(X), 1)), X])
        A = Xb.T @ Xb + self.alpha * np.eye(Xb.shape[1])
        A[0, 0] -= self.alpha                        # don't penalize the bias
        self.w = np.linalg.solve(A, Xb.T @ y)
        return self

    def predict(self, X):
        Xb = np.hstack([np.ones((len(np.asarray(X, float)), 1)), np.asarray(X, float)])
        return Xb @ self.w

def demo():
    np.random.seed(SEED)
    from sklearn.datasets import make_classification, make_friedman1

    # ---------- classification ----------
    X, y = make_classification(n_samples=500, n_features=10, n_informative=6,
                               n_redundant=2, n_classes=3, n_clusters_per_class=1,
                               random_state=SEED)
    Xtr, ytr, Xte, yte = X[:380], y[:380], X[380:], y[380:]

    bases = [lambda: _Tree(task="classification", max_depth=4),
             lambda: _LogReg(lr=0.3, n_iter=400)]
    for f, name in [(bases[0], "tree"), (bases[1], "logreg")]:
        m = f().fit(Xtr, ytr)
        print(f"[clf] base {name:7s} acc={np.mean(m.predict(Xte) == yte):.3f}")

    stack = StackingNumPy(bases, lambda: _LogReg(lr=0.3, n_iter=400),
                          task="classification", n_folds=5).fit(Xtr, ytr)
    print(f"[clf] STACK (OOF) acc={np.mean(stack.predict(Xte) == yte):.3f}")
    stack_pt = StackingNumPy(bases, lambda: _LogReg(lr=0.3, n_iter=400),
                             task="classification", passthrough=True).fit(Xtr, ytr)
    print(f"[clf] STACK +passthrough acc={np.mean(stack_pt.predict(Xte) == yte):.3f}")
    sk = sklearn_reference(Xtr, ytr)
    print(f"[clf] sklearn Stacking acc={np.mean(sk.predict(Xte) == yte):.3f}")

    # ---------- regression ----------
    Xr, yr = make_friedman1(n_samples=400, noise=1.0, random_state=SEED)
    Xrtr, yrtr, Xrte, yrte = Xr[:300], yr[:300], Xr[300:], yr[300:]
    rbases = [lambda: _Tree(task="regression", max_depth=4),
              lambda: _Ridge(alpha=1.0)]
    stackr = StackingNumPy(rbases, lambda: _Ridge(alpha=1.0),
                           task="regression", n_folds=5).fit(Xrtr, yrtr)
    print(f"[reg] STACK MSE={np.mean((stackr.predict(Xrte) - yrte) ** 2):.3f}")
    skr = sklearn_reference(Xrtr, yrtr, task="regression")
    print(f"[reg] sklearn Stacking MSE={np.mean((skr.predict(Xrte) - yrte) ** 2):.3f}")


class StackingNumPy:
    """Stacked generalization.

    base_factories : list of 0-arg callables -> fresh base learners.
    meta_factory   : 0-arg callable -> the meta learner.
    """

    def __init__(self, base_factories, meta_factory, task="classification",
                 n_folds=5, passthrough=False, seed=SEED):
        self.base_factories = base_factories
        self.meta_factory = meta_factory
        self.task = task
        self.n_folds = n_folds
        self.passthrough = passthrough
        self.seed = seed

    def _base_out(self, model, X):
        """Per-base output used as meta-features: probabilities (clf, drop last
        column to avoid collinearity) or scalar prediction (reg)."""
        if self.task == "classification":
            P = model.predict_proba(X)
            return P[:, :-1]                # K-1 columns suffice
        return model.predict(X).reshape(-1, 1)

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y)
        n = len(X)
        rng = np.random.default_rng(self.seed)
        self.classes_ = np.unique(y) if self.task == "classification" else None

        folds = _kfold_indices(n, self.n_folds, rng)
        # ---- generate OUT-OF-FOLD meta-features (no leakage) ----
        meta_cols = []
        for factory in self.base_factories:
            col = np.zeros((n, (len(self.classes_) - 1) if self.task == "classification" else 1))
            for val in folds:
                train = np.setdiff1d(np.arange(n), val)
                m = copy.deepcopy(factory())
                m.fit(X[train], y[train])           # fit on K-1 folds
                col[val] = self._base_out(m, X[val])  # predict the held-out fold
            meta_cols.append(col)
        Z = np.hstack(meta_cols)
        if self.passthrough:
            Z = np.hstack([Z, X])

        # ---- meta-learner trained on OOF features ----
        self.meta_ = copy.deepcopy(self.meta_factory())
        self.meta_.fit(Z, y)

        # ---- refit each base model on the FULL training set for test time ----
        self.bases_ = [copy.deepcopy(f()).fit(X, y) for f in self.base_factories]
        return self

    def _meta_features(self, X):
        cols = [self._base_out(m, X) for m in self.bases_]
        Z = np.hstack(cols)
        if self.passthrough:
            Z = np.hstack([Z, np.asarray(X, float)])
        return Z

    def predict(self, X):
        return self.meta_.predict(self._meta_features(X))

    def predict_proba(self, X):
        assert self.task == "classification"
        return self.meta_.predict_proba(self._meta_features(X))

## 5. Reference / cross-check — why not PyTorch?

Stacking is an *orchestration* layer: it cross-validates arbitrary base learners
(here a shallow tree, logistic regression, ridge) to build OOF meta-features and
fits a meta-learner. The orchestration has nothing to differentiate end-to-end,
so an idiomatic PyTorch model is not the natural tool. We cross-check against
scikit-learn's `Stacking*` estimators.

In [ ]:
# ===== actual implementation from stacking.py =====
def sklearn_reference(X, y, task="classification", **kw):
    from sklearn.linear_model import LogisticRegression, Ridge
    from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
    if task == "classification":
        from sklearn.ensemble import StackingClassifier
        estimators = [("stump", DecisionTreeClassifier(max_depth=1)),
                      ("tree", DecisionTreeClassifier(max_depth=4))]
        return StackingClassifier(estimators,
                                  final_estimator=LogisticRegression(max_iter=500),
                                  cv=5, **kw).fit(X, y)
    from sklearn.ensemble import StackingRegressor
    estimators = [("stump", DecisionTreeRegressor(max_depth=1)),
                  ("tree", DecisionTreeRegressor(max_depth=4))]
    return StackingRegressor(estimators, final_estimator=Ridge(), cv=5, **kw).fit(X, y)

## 6. Train — base models vs the stacked ensemble, and the cross-check

In [ ]:
demo()

## 7. Visualization — base vs stacked decision regions

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_moons
import stacking as M

X, y = make_moons(n_samples=300, noise=0.25, random_state=0)
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-.5, X[:,0].max()+.5, 200),
                     np.linspace(X[:,1].min()-.5, X[:,1].max()+.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

bases = [lambda: M._Tree(task="classification", max_depth=3),
         lambda: M._LogReg(lr=0.5, n_iter=500)]
models = {
    "tree (base)": bases[0]().fit(X, y),
    "logreg (base)": bases[1]().fit(X, y),
    "stacked": M.StackingNumPy(bases, lambda: M._LogReg(lr=0.5, n_iter=500),
                               task="classification", passthrough=True).fit(X, y),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models.items()):
    zz = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.3, cmap="coolwarm")
    ax.scatter(X[:,0], X[:,1], c=y, s=12, edgecolor="k", cmap="coolwarm")
    ax.set_title(name)
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Always** build meta-features out-of-fold; in-sample base predictions leak and
  ruin the meta-learner.
- Keep the **meta-learner simple** (linear / ridge / logistic) to avoid
  overfitting the small set of meta-features; let the bases be diverse and strong.
- Diversity matters more than individual accuracy — decorrelated bases give the
  blender something to exploit.
- More expensive than voting (it refits bases $K{+}1$ times). When the bases are
  near-identical, plain **soft voting** is simpler and nearly as good.